# XDM / MFEX Practical Colab: полный рабочий пример без обучения модели

Датасет: **Breast Cancer Wisconsin Diagnostic**.

Этот Colab показывает полный pipeline, который студент должен повторить на своих двух датасетах:

1. один медицинский датасет;
2. один ESG / экономический датасет.

## Строгое правило

В этом ноутбуке нет обучения предсказательной модели.  
Не используется kNN-классификатор, Random Forest, Logistic Regression, XGBoost, нейросеть, эпохи обучения или SHAP/LIME поверх модели.

Валидация делается через **MFF-NoFit**: проверку того, что consensus top-k признаки лучше случайных k признаков сохраняют структуру target в пространстве данных.

## 0. Импорт библиотек

Используются только базовые научные библиотеки и `sklearn` для загрузки датасета, разделения и mutual information. Предиктивные estimator-объекты не создаются.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import pairwise_distances
from scipy.stats import spearmanr

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True

## 1. Загрузка данных и EDA

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
target_col = 'target'

print('Shape:', df.shape)
display(df.head())
print('\\nTarget distribution:')
display(df[target_col].value_counts(normalize=True).rename('share'))
print('\\nMissing values:')
display(df.isna().sum().sort_values(ascending=False).head(10))
display(df.describe().T.head(10))

## 2. Ручная предобработка без estimator-объектов

В этом примере все признаки числовые. Для универсальности оставлен код:

- заполнение пропусков медианой;
- one-hot encoding категорий;
- ручная стандартизация через среднее и стандартное отклонение.

Это не обучение модели, а подготовка матрицы данных к расстояниям и сравнениям.

In [ ]:
def preprocess_table(df, target_col):
    data = df.copy()
    y = data[target_col].astype(int).reset_index(drop=True)
    X = data.drop(columns=[target_col])

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    for col in num_cols:
        X[col] = X[col].fillna(X[col].median())

    for col in cat_cols:
        mode_values = X[col].mode(dropna=True)
        fill_value = mode_values.iloc[0] if len(mode_values) else 'missing'
        X[col] = X[col].fillna(fill_value)

    if cat_cols:
        X = pd.get_dummies(X, columns=cat_cols, drop_first=False)

    X = X.reset_index(drop=True)
    means = X.mean(axis=0)
    stds = X.std(axis=0).replace(0, 1.0)
    X_scaled = (X - means) / stds
    return X_scaled, y, list(X_scaled.columns)

X, y, feature_names = preprocess_table(df, target_col)

X_work, X_holdout, y_work, y_holdout = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

X_work = X_work.reset_index(drop=True)
X_holdout = X_holdout.reset_index(drop=True)
y_work = y_work.reset_index(drop=True)
y_holdout = y_holdout.reset_index(drop=True)

print('Work set:', X_work.shape)
print('Holdout set:', X_holdout.shape)

## 3. Вспомогательные функции

In [ ]:
def minmax_normalize(series):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
    s = s - s.min()
    if s.max() == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return s / s.max()


def rank_from_importance(series):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
    return s.rank(ascending=False, method='average')


def top_barplot(series, title, top_n=10):
    s = pd.Series(series).sort_values(ascending=True).tail(top_n)
    ax = s.plot(kind='barh')
    ax.set_title(title)
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

## 4. Метод 1: Statistical Importance = Mutual Information + Relief-style contrast

In [ ]:
def relief_style_contrast(X, y, n_neighbors=10):
    X_arr = np.asarray(X, dtype=float)
    y_arr = np.asarray(y)
    n, p = X_arr.shape
    distances = pairwise_distances(X_arr, X_arr)
    np.fill_diagonal(distances, np.inf)
    scores = np.zeros(p)

    for i in range(n):
        same_idx = np.where(y_arr == y_arr[i])[0]
        diff_idx = np.where(y_arr != y_arr[i])[0]
        same_idx = same_idx[same_idx != i]
        if len(same_idx) == 0 or len(diff_idx) == 0:
            continue
        hit_idx = same_idx[np.argsort(distances[i, same_idx])[:n_neighbors]]
        miss_idx = diff_idx[np.argsort(distances[i, diff_idx])[:n_neighbors]]
        hit_diff = np.mean(np.abs(X_arr[i] - X_arr[hit_idx]), axis=0)
        miss_diff = np.mean(np.abs(X_arr[i] - X_arr[miss_idx]), axis=0)
        scores += miss_diff - hit_diff

    return pd.Series(scores / max(n, 1), index=X.columns)

mi_values = mutual_info_classif(X_work, y_work, random_state=RANDOM_STATE)
mi_importance = pd.Series(mi_values, index=feature_names, name='MI')
relief_importance = relief_style_contrast(X_work, y_work, n_neighbors=10)

stat_importance = (
    minmax_normalize(mi_importance) + minmax_normalize(relief_importance)
) / 2
stat_importance.name = 'Stat_MI_ReliefStyle'

top_barplot(stat_importance, 'Statistical importance: MI + Relief-style contrast')

## 5. Метод 2: Geometric Importance без модели

Score признака: отношение средней дистанции между классами к средней дистанции внутри классов.

In [ ]:
def geometric_importance_univariate(X, y):
    scores = {}
    y_arr = np.asarray(y)
    for col in X.columns:
        values = np.asarray(X[col], dtype=float).reshape(-1, 1)
        d = pairwise_distances(values, values)
        same = y_arr[:, None] == y_arr[None, :]
        diff = ~same
        np.fill_diagonal(same, False)
        within = d[same].mean() if np.any(same) else 0.0
        between = d[diff].mean() if np.any(diff) else 0.0
        scores[col] = between / (within + 1e-9)
    return pd.Series(scores)

geom_importance = geometric_importance_univariate(X_work, y_work)
geom_importance = minmax_normalize(geom_importance)
geom_importance.name = 'Geom_DistanceSeparation'

top_barplot(geom_importance, 'Geometric importance: between/within distance separation')

## 6. Метод 3: Causal-screening Importance без causal-model object

Здесь используется не обучаемая causal-модель, а conditional-dependence screening через partial correlation.

Для каждого признака:

1. считаем связь с target без контроля;
2. контролируем 1–2 наиболее связанных с ним признака;
3. берём среднюю устойчивую абсолютную partial correlation.

Это осторожнее называть **causal-screening**, а не доказанной причинностью.

In [ ]:
def partial_corr_from_matrix(frame, x_col, y_col, z_cols=None):
    z_cols = z_cols or []
    cols = [x_col, y_col] + list(z_cols)
    arr = frame[cols].to_numpy(dtype=float)
    corr = np.corrcoef(arr, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    precision = np.linalg.pinv(corr)
    denom = np.sqrt(max(precision[0, 0] * precision[1, 1], 1e-12))
    return float(-precision[0, 1] / denom)


def causal_screening_importance(X, y, max_controls=2):
    frame = X.copy()
    frame['__target__'] = np.asarray(y, dtype=float)
    feature_cols = list(X.columns)
    scores = {}

    corr_abs = frame[feature_cols].corr().abs().fillna(0.0)

    for col in feature_cols:
        candidate_controls = corr_abs[col].sort_values(ascending=False).index.tolist()
        candidate_controls = [c for c in candidate_controls if c != col]
        controls = candidate_controls[:max_controls]

        values = []
        values.append(abs(partial_corr_from_matrix(frame, col, '__target__', [])))
        for m in range(1, len(controls) + 1):
            values.append(abs(partial_corr_from_matrix(frame, col, '__target__', controls[:m])))

        # Устойчивость: среднее значение, но штрафуем сильное падение связи.
        values = np.asarray(values, dtype=float)
        stability = values.min() / (values.max() + 1e-9)
        scores[col] = values.mean() * stability

    return pd.Series(scores)

causal_importance = causal_screening_importance(X_work, y_work, max_controls=2)
causal_importance = minmax_normalize(causal_importance)
causal_importance.name = 'CausalScreen_PartialCorr'

top_barplot(causal_importance, 'Causal-screening importance: stable partial dependence')

## 7. Метод 4: Structural Importance через правила подгрупп

Генерируем простые правила по квантильным интервалам признаков и считаем WRAcc. Важность признака равна сумме качества top-правил, где он участвует.

In [ ]:
def make_quantile_bins(series, q=4):
    try:
        return pd.qcut(series, q=q, duplicates='drop')
    except Exception:
        return pd.cut(series, bins=q, duplicates='drop')


def subgroup_rules_importance(X, y, n_bins=4, top_rules=40, min_coverage=0.05):
    y_arr = np.asarray(y).astype(int)
    base_rate = y_arr.mean()
    rules = []

    binned = pd.DataFrame(index=X.index)
    for col in X.columns:
        binned[col] = make_quantile_bins(X[col], q=n_bins)

    # Single-feature rules
    for col in X.columns:
        for level in binned[col].dropna().unique():
            mask = (binned[col] == level).to_numpy()
            coverage = mask.mean()
            if coverage < min_coverage:
                continue
            rate = y_arr[mask].mean()
            wracc = coverage * abs(rate - base_rate)
            rules.append({
                'features': (col,),
                'rule': f'{col} in {level}',
                'coverage': coverage,
                'target_rate': rate,
                'wracc': wracc
            })

    # Pairwise rules among top statistical candidates to keep runtime low
    candidate_cols = stat_importance.sort_values(ascending=False).head(8).index.tolist()
    for i, col_a in enumerate(candidate_cols):
        for col_b in candidate_cols[i+1:]:
            for lev_a in binned[col_a].dropna().unique():
                for lev_b in binned[col_b].dropna().unique():
                    mask = ((binned[col_a] == lev_a) & (binned[col_b] == lev_b)).to_numpy()
                    coverage = mask.mean()
                    if coverage < min_coverage:
                        continue
                    rate = y_arr[mask].mean()
                    wracc = coverage * abs(rate - base_rate)
                    rules.append({
                        'features': (col_a, col_b),
                        'rule': f'{col_a} in {lev_a} AND {col_b} in {lev_b}',
                        'coverage': coverage,
                        'target_rate': rate,
                        'wracc': wracc
                    })

    rules_df = pd.DataFrame(rules).sort_values('wracc', ascending=False).head(top_rules)
    importance = pd.Series(0.0, index=X.columns)
    for _, row in rules_df.iterrows():
        for feat in row['features']:
            importance.loc[feat] += row['wracc']

    return importance, rules_df

struct_importance, top_rules_df = subgroup_rules_importance(X_work, y_work, n_bins=4, top_rules=40)
struct_importance = minmax_normalize(struct_importance)
struct_importance.name = 'Structural_SubgroupRules'

display(top_rules_df.head(10))
top_rules_df.to_csv('xdm_top_rules.csv', index=False)
top_barplot(struct_importance, 'Structural importance: subgroup rules')

## 8. Консенсусное ранжирование

In [ ]:
importance_df = pd.DataFrame({
    'Stat_MI_ReliefStyle': stat_importance,
    'Geom_DistanceSeparation': geom_importance,
    'CausalScreen_PartialCorr': causal_importance,
    'Structural_SubgroupRules': struct_importance
})

method_cols = list(importance_df.columns)
rank_df = importance_df.apply(rank_from_importance, axis=0)
rank_df['median_rank'] = rank_df[method_cols].median(axis=1)
rank_df['mean_rank'] = rank_df[method_cols].mean(axis=1)
rank_df['rank_iqr'] = rank_df[method_cols].quantile(0.75, axis=1) - rank_df[method_cols].quantile(0.25, axis=1)

threshold = rank_df['rank_iqr'].quantile(0.50)
rank_df['stable'] = rank_df['rank_iqr'] <= threshold
if rank_df['stable'].sum() < 5:
    threshold = rank_df['rank_iqr'].quantile(0.75)
    rank_df['stable'] = rank_df['rank_iqr'] <= threshold

rank_df = rank_df.sort_values(['median_rank', 'rank_iqr'])

print('IQR stability threshold:', round(threshold, 4))
print('Stable features:', int(rank_df['stable'].sum()))
display(rank_df.head(15))
rank_df.to_csv('xdm_feature_importance_ranks.csv')

## 9. Визуализация согласованности методов

In [ ]:
corr = rank_df[method_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, vmin=-1, vmax=1)
ax.set_xticks(range(len(method_cols)))
ax.set_yticks(range(len(method_cols)))
ax.set_xticklabels(method_cols, rotation=45, ha='right')
ax.set_yticklabels(method_cols)
for i in range(len(method_cols)):
    for j in range(len(method_cols)):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center')
ax.set_title('Spearman agreement between XDM methods')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

rank_df.head(10)['median_rank'].sort_values(ascending=False).plot(kind='barh')
plt.title('Top-10 consensus features')
plt.xlabel('Median rank; lower is better')
plt.tight_layout()
plt.show()

## 10. MFF-NoFit: валидация без классификатора

Score подмножества признаков:

$$
S(F)=rac{mean\ distance\ between\ classes}{mean\ distance\ within\ classes}
$$

MFF-NoFit:

$$
MFF(k)=S(top	ext{-}k)-mean(S(random\ k))
$$

Это проверяет, что consensus-признаки лучше случайных сохраняют структуру target.

In [ ]:
def separation_score(X_subset, y):
    X_arr = np.asarray(X_subset, dtype=float)
    y_arr = np.asarray(y)
    d = pairwise_distances(X_arr, X_arr)
    same = y_arr[:, None] == y_arr[None, :]
    diff = ~same
    np.fill_diagonal(same, False)
    within = d[same].mean() if np.any(same) else 0.0
    between = d[diff].mean() if np.any(diff) else 0.0
    return float(between / (within + 1e-9))


def compute_mff_nofit(rank_df, X_eval, y_eval, k, n_random=200, random_state=42):
    rng = np.random.default_rng(random_state)
    all_features = list(rank_df.index)
    k = min(k, len(all_features))
    top_features = rank_df.head(k).index.tolist()

    top_score = separation_score(X_eval[top_features], y_eval)
    random_scores = []
    for _ in range(n_random):
        random_features = rng.choice(all_features, size=k, replace=False).tolist()
        random_scores.append(separation_score(X_eval[random_features], y_eval))

    random_scores = np.asarray(random_scores)
    p_value = (np.sum(random_scores >= top_score) + 1) / (len(random_scores) + 1)
    return {
        'k': k,
        'top_score': top_score,
        'random_mean': float(random_scores.mean()),
        'random_std': float(random_scores.std()),
        'MFF_NoFit': float(top_score - random_scores.mean()),
        'permutation_p_value': float(p_value),
        'top_features': ', '.join(top_features)
    }

k_values = [5, 10, 15]
stable_count = int(rank_df['stable'].sum())
if stable_count > 0:
    k_values.append(stable_count)
k_values = sorted(set([min(k, len(feature_names)) for k in k_values]))

mff_rows = []
for k in k_values:
    mff_rows.append(compute_mff_nofit(rank_df, X_holdout, y_holdout, k=k, n_random=200, random_state=RANDOM_STATE + k))

mff_df = pd.DataFrame(mff_rows)
display(mff_df)
mff_df.to_csv('xdm_mff_nofit_results.csv', index=False)

mff_df.plot(x='k', y=['top_score', 'random_mean'], kind='bar')
plt.title('MFF-NoFit validation: consensus top-k vs random-k')
plt.ylabel('Distance separation score')
plt.tight_layout()
plt.show()

mff_df.plot(x='k', y='MFF_NoFit', kind='bar', legend=False)
plt.title('MFF-NoFit by k')
plt.ylabel('MFF-NoFit')
plt.tight_layout()
plt.show()

## 11. Stability check через bootstrap рангов

Повторяем расчёт importance на bootstrap-подвыборках и смотрим, как часто признак входит в top-10 consensus.

In [ ]:
def bootstrap_consensus_stability(X, y, n_bootstrap=30, sample_frac=0.80, random_state=42):
    rng = np.random.default_rng(random_state)
    n = len(X)
    rank_runs = []

    for b in range(n_bootstrap):
        idx = rng.choice(np.arange(n), size=int(sample_frac * n), replace=True)
        Xb = X.iloc[idx].reset_index(drop=True)
        yb = y.iloc[idx].reset_index(drop=True)

        mib = pd.Series(mutual_info_classif(Xb, yb, random_state=random_state + b), index=X.columns)
        reliefb = relief_style_contrast(Xb, yb, n_neighbors=10)
        statb = (minmax_normalize(mib) + minmax_normalize(reliefb)) / 2
        geomb = minmax_normalize(geometric_importance_univariate(Xb, yb))
        causalb = minmax_normalize(causal_screening_importance(Xb, yb, max_controls=2))
        structb, _ = subgroup_rules_importance(Xb, yb, n_bins=4, top_rules=30)
        structb = minmax_normalize(structb)

        temp = pd.DataFrame({
            'stat': statb,
            'geom': geomb,
            'causal': causalb,
            'struct': structb
        }).apply(rank_from_importance, axis=0)
        consensus = temp.median(axis=1).rank(ascending=True, method='average')
        rank_runs.append(consensus.rename(f'boot_{b}'))

    boot = pd.concat(rank_runs, axis=1)
    summary = pd.DataFrame({
        'bootstrap_mean_rank': boot.mean(axis=1),
        'bootstrap_std_rank': boot.std(axis=1),
        'top10_frequency': (boot <= 10).mean(axis=1)
    }).sort_values(['bootstrap_mean_rank', 'bootstrap_std_rank'])
    return boot, summary

boot_ranks, boot_summary = bootstrap_consensus_stability(X_work, y_work, n_bootstrap=30)
display(boot_summary.head(15))
boot_summary.to_csv('xdm_bootstrap_stability.csv')

## 12. Минимальный Streamlit dashboard

Эта ячейка сохраняет файл `xdm_streamlit_app.py`.

Запуск:

```bash
streamlit run xdm_streamlit_app.py
```

Dashboard использует уже сохранённые CSV и не обучает моделей.

In [ ]:
app_code = """
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title='XDM / MFEX No-Fit Dashboard', layout='wide')
st.title('XDM / MFEX No-Fit Dashboard')
st.caption('Model-free explanation of tabular data: ranks, consensus, stability, MFF-NoFit')

rank_file = st.file_uploader('Upload xdm_feature_importance_ranks.csv', type=['csv'])
mff_file = st.file_uploader('Upload xdm_mff_nofit_results.csv', type=['csv'])

if rank_file is None:
    st.info('Upload rank CSV to start.')
    st.stop()

rank_df = pd.read_csv(rank_file, index_col=0)
method_cols = [c for c in rank_df.columns if c not in ['median_rank', 'mean_rank', 'rank_iqr', 'stable']]

left, right = st.columns([1, 2])
with left:
    selected = st.selectbox('Rank source', method_cols + ['median_rank'])
    max_iqr = float(rank_df['rank_iqr'].max())
    threshold = st.slider('IQR stability threshold', 0.0, max_iqr, float(rank_df['rank_iqr'].median()))

view = rank_df.copy()
view['stable_by_slider'] = view['rank_iqr'] <= threshold
view = view.sort_values([selected, 'rank_iqr'])

with right:
    st.subheader('Top consensus / method features')
    st.dataframe(view.head(20), use_container_width=True)
    top = view.head(10)[selected].sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 5))
    top.plot(kind='barh', ax=ax)
    ax.set_title(f'Top-10 by {selected}; lower rank is better')
    ax.set_xlabel('Rank')
    st.pyplot(fig)

st.subheader('Stable features')
st.dataframe(view[view['stable_by_slider']].head(30), use_container_width=True)

if mff_file is not None:
    mff = pd.read_csv(mff_file)
    st.subheader('MFF-NoFit')
    st.dataframe(mff, use_container_width=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    mff.plot(x='k', y='MFF_NoFit', kind='bar', ax=ax, legend=False)
    ax.set_title('MFF-NoFit by k')
    ax.set_ylabel('MFF-NoFit')
    st.pyplot(fig)
"""

with open('xdm_streamlit_app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('Saved: xdm_streamlit_app.py')

## 13. Что студент сдаёт по каждому датасету

Минимальный набор файлов:

1. заполненный `.ipynb`;
2. `xdm_feature_importance_ranks.csv`;
3. `xdm_mff_nofit_results.csv`;
4. `xdm_top_rules.csv`;
5. `xdm_bootstrap_stability.csv`;
6. короткий текстовый вывод: 7–10 предложений.

## 14. Вопросы для защиты

1. Почему этот pipeline считается model-free?
2. Почему MI не является причинностью?
3. Что именно измеряет геометрическая важность?
4. Почему causal-screening нельзя называть доказанной причинностью?
5. Что означает высокий IQR рангов?
6. Почему MFF-NoFit лучше подходит под строгий запрет на обучение, чем 1-NN accuracy?
7. Почему top-k consensus может проиграть случайным признакам?
8. Какие признаки требуют доменной проверки врачом или ESG-экспертом?